<h1 style = "color : #0EE071; text-align : center;"><em>Nata Project</em> - Modelling Notebook</h1>
<p style = "font-size : 16px; text-align: center;">This notebook focuses on training, tuning, and evaluating various machine learning models to find the best performing solution.</p>
<br>
<p style = "font-size : 12px; text-align: center;"><b>NOVA IMS</b></p>
<p style = "font-size : 10px; text-align: center;">Machine Learning I</p>
<p style = "font-size : 10px; text-align: center;">Group 08: Diogo Gonçalves, João Marques, Juan Mendes, Gustavo Franco & Lucas Casimiro</p>
<br>

## <a class="anchor" id="0th-bullet">Table of Contents</a>

* [<b>1. Imports & Dataset Loading</b>](#1st-bullet)<br>
* [<b>2. Logistic Regression</b>](#2nd-bullet)<br>
* [<b>3. Random Forest</b>](#3rd-bullet)<br>
* [<b>4. Gradient Boosting</b>](#4th-bullet)<br>
* [<b>5. K-Nearest Neighbors (KNN)</b>](#5th-bullet)<br>
* [<b>6. Decision Trees</b>](#6th-bullet)<br>
* [<b>7. MLP Classifier</b>](#7th-bullet)<br>
* [<b>8. AdaBoost Classifier</b>](#8th-bullet)<br>
    * [<b>8.1 AdaBoost Hyperparameter Optimization</b>](#9th-bullet)<br>
* [<b>9. Emsemble Methods</b>](#8th-bullet)<br>
    * [<b>9.1 Final Stacking Classifier</b>](#9th-bullet)<br>

<hr style = "border: 3px solid #0EE071;">
<h2  style = "color : #0EE071;"> 1. Imports & Dataset Loading</h2>

In [4]:
import pandas as pd

import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.model_selection import GridSearchCV, ParameterGrid
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

import tqdm
from tqdm import tqdm
from tqdm.auto import tqdm

import joblib

import contextlib
from contextlib import contextmanager

import warnings
warnings.filterwarnings("ignore")


<p style = "font-size : 15px;">In this section, we import the necessary libraries and load the final datasets <code>X_train_final</code> and <code>y_train_final</code> prepared in the previous notebook.</p>

In [23]:
X_train_final=pd.read_pickle('Nata_files/data_clean/X_train_final.pkl')
X_test_final=pd.read_pickle('Nata_files/data_clean/X_test_final.pkl') 

In [24]:
y_train_final = pd.read_pickle('Nata_files/data_clean/y_train_clean.pkl')
y_test_final = pd.read_pickle('Nata_files/data_clean/y_test_clean.pkl')

<hr style = "border: 3px solid #0EE071;"> <h2 style = "color : #0EE071;">Custom Progress Tracking</a></h2> <p style = "font-size : 15px;">A utility function designed to integrate <code>tqdm</code> progress bars with <code>joblib</code> parallel processing.</p>

<p>This implementation handles the following core mechanics: </p> <ul style = "font-size : 15px;"> <li><code>@contextlib.contextmanager</code>: This decorator allows the function to be used with the <code>with</code> statement, ensuring a clean setup and teardown process.</li> <li><code>TqdmBatchCompletionCallback</code>: A custom class that overrides joblib's internal callback mechanism to trigger a progress update every time a batch of tasks completes.</li> <li><code>Monkey Patching</code>: The function temporarily swaps joblib's global callback with our custom one and uses a <code>finally</code> block to ensure the original behavior is restored even if an error occurs.</li> </ul>

In [8]:
# This context manager patches joblib to report progress to tqdm
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """Context manager to patch joblib to report into tqdm progress bar given as argument"""
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()

<hr style = "border: 3px solid #0EE071;">
<h2 style = "color : #0EE071;">2. Logistic Regression</a></h2>
<p style = "font-size : 15px;">Baseline modeling using Logistic Regression to establish a performance benchmark.</p>
<p style = "font-size : 15px;">
    We begin our modeling phase with <b>Logistic Regression</b>. Despite its name, this is a classification algorithm used to predict binary outcomes by estimating the probability that an instance belongs to a particular class.
</p>
<p style = "font-size : 15px;">
    In this implementation, we defined several key attributes to ensure stability and convergence:
</p>
<ul style = "font-size : 15px;">
    <li><code>solver='lbfgs'</code>: The mathematical "engine" used to find the best-fitting line for our data. It is the default choice for most small to medium datasets.</li>
    <li><code>max_iter=1000</code>: This tells the model how many times it can iterate (try to improve) to find the optimal solution. We increased this to 1000 to ensure the model has enough "time" to converge.</li>
    <li><code>random_state=42</code>: A seed that ensures our results are reproducible. Every time we run this cell, the model will produce the exact same results.<b>(This will be used on all models)</b></li>
</ul>

In [9]:
log_reg = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
log_reg.fit(X_train_final, y_train_final)

y_pred_lr = log_reg.predict(X_test_final)
print("Logistic Regression Accuracy:", accuracy_score(y_test_final, y_pred_lr))
print(classification_report(y_test_final, y_pred_lr))

Logistic Regression Accuracy: 0.7423076923076923
              precision    recall  f1-score   support

           0       0.68      0.55      0.61       379
           1       0.77      0.85      0.81       661

    accuracy                           0.74      1040
   macro avg       0.72      0.70      0.71      1040
weighted avg       0.74      0.74      0.74      1040



<p>Here <code>max_iter</code> is lower to verify if the value changes drastically or not

In [ ]:
log_reg2 = LogisticRegression(max_iter=500, solver='lbfgs', random_state=42)
log_reg2.fit(X_train_final, y_train_final)

y_pred_lr = log_reg2.predict(X_test_final)
print("Logistic Regression Accuracy:", accuracy_score(y_test_final, y_pred_lr))
print(classification_report(y_test_final, y_pred_lr))

Logistic Regression Accuracy: 0.7423076923076923
              precision    recall  f1-score   support

           0       0.68      0.55      0.61       379
           1       0.77      0.85      0.81       661

    accuracy                           0.74      1040
   macro avg       0.72      0.70      0.71      1040
weighted avg       0.74      0.74      0.74      1040



<hr style = "border: 3px solid #0EE071;">
<h2 style = "color : #0EE071;">3. Random Forest</a></h2>
<p style = "font-size : 15px;">Implementing Random Forest with hyperparameter tuning via <code>GridSearchCV</code> to improve accuracy and handle non-linear relationships, testing parameters to have the best results possible.</p>
<p>We have configured this forest with specific attributes to balance its learning power:</p>
</p>
<ul style = "font-size : 15px;">
    <li><code>n_estimators=100</code>: This defines the size of our forest. We are using 100 individual decision trees to make the final prediction.</li>
    <li><code>max_depth=10</code>: This limits how deep each individual tree can grow. By stopping the trees at 10 levels, we prevent them from becoming too complex and "memorizing" the training data (overfitting).</li>
    
</ul>

In [11]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10,random_state=42)
rf.fit(X_train_final, y_train_final)

y_pred_rf = rf.predict(X_test_final)
print("Random Forest Accuracy:", accuracy_score(y_test_final, y_pred_rf))
print(classification_report(y_test_final, y_pred_rf))

Random Forest Accuracy: 0.775
              precision    recall  f1-score   support

           0       0.73      0.61      0.66       379
           1       0.80      0.87      0.83       661

    accuracy                           0.78      1040
   macro avg       0.76      0.74      0.75      1040
weighted avg       0.77      0.78      0.77      1040



<p>Here we varied the <code>n_estimators</code>to a lower size of forest to verify changes

In [12]:
rf2 = RandomForestClassifier( n_estimators=10, max_depth=10,random_state=42)
rf2.fit(X_train_final, y_train_final)

y_pred_rf2 = rf2.predict(X_test_final)
print("Random Forest Accuracy:", accuracy_score(y_test_final, y_pred_rf2))
print(classification_report(y_test_final, y_pred_rf2))

Random Forest Accuracy: 0.7615384615384615
              precision    recall  f1-score   support

           0       0.71      0.59      0.64       379
           1       0.78      0.86      0.82       661

    accuracy                           0.76      1040
   macro avg       0.75      0.72      0.73      1040
weighted avg       0.76      0.76      0.76      1040



In [13]:
rf3=RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)
rf3.fit(X_train_final, y_train_final)

y_pred_rf3 = rf3.predict(X_test_final)
print("Random Forest Accuracy:", accuracy_score(y_test_final, y_pred_rf3))
print(classification_report(y_test_final, y_pred_rf3))

Random Forest Accuracy: 0.7846153846153846
              precision    recall  f1-score   support

           0       0.74      0.63      0.68       379
           1       0.80      0.87      0.84       661

    accuracy                           0.78      1040
   macro avg       0.77      0.75      0.76      1040
weighted avg       0.78      0.78      0.78      1040



In [ ]:
rf4=RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
rf4.fit(X_train_final, y_train_final)

y_pred_rf4 = rf4.predict(X_test_final)
print("Random Forest Accuracy:", accuracy_score(y_test_final, y_pred_rf4))
print(classification_report(y_test_final, y_pred_rf4))

Random Forest Accuracy: 0.7682692307692308
              precision    recall  f1-score   support

           0       0.72      0.60      0.65       379
           1       0.79      0.87      0.83       661

    accuracy                           0.77      1040
   macro avg       0.75      0.73      0.74      1040
weighted avg       0.76      0.77      0.76      1040



<p>After some testing we do a <code>Grid Search</code>to achieve the pest values for the corresponding parameters of <code>Random Forest</code> introducing <code>min_samples_split</code> and <code>min_samples_leaf</code></p>

In [ ]:
rf = RandomForestClassifier(random_state=42)

# 2. Define the parameter grid
param_grid_rf = {
    # Number of trees in the forest. More is usually better, but slower.
    'n_estimators': [10,20,50, 100, 200],
    
    # Maximum depth of the tree.
    # None = split until leaves are pure (can overfit).
    # 10, 20 = limits depth to prevent overfitting.
    'max_depth': [None, 10, 20, 30],
    
    'criterion': ['gini', 'entropy', 'log_loss'],
    
    # Minimum number of samples required to split a node.
    # Higher numbers (e.g., 5, 10) prevent the model from learning "noise".
    'min_samples_split': [2, 5, 10],
    
    # Minimum number of samples required at each leaf node.
    'min_samples_leaf': [1, 2, 4],
    
    'max_features': ['auto', 'sqrt', 'log2'] # Number of features to consider at every split
}

# 3. Setup Grid Search
grid_search_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=5,
    scoring='accuracy', # or 'f1_macro', 'recall', etc.
    n_jobs=-1,          # Use all CPU cores
    verbose=2
)

# 4. Fit to your data
grid_search_rf.fit(X_train_final, y_train_final)

# 5. Results
print("Best RF Parameters:", grid_search_rf.best_params_)
print("Best Score:", grid_search_rf.best_score_)

Fitting 5 folds for each of 1620 candidates, totalling 8100 fits
Best RF Parameters: {'criterion': 'gini', 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best Score: 0.7794105225400353


<p style = "font-size : 15px;">With the optimization complete, we arrived at the following optimal parameters for our model:</p>
<ul style = "font-size : 15px;">
    <li><code>'criterion': gini</code></li>
    <li><code>'max_depth': 20</code></li>
    <li><code>'min_samples_leaf': 1</code></li>
    <li><code>'min_samples_split': 2</code></li>
    <li><code>'n_estimators': 200</code></li>
    <li><code>'max_features': sqrt</code></li>
</ul>

<hr style = "border: 3px solid #0EE071;">
<h2 style = "color : #0EE071;">4. Gradient Boosting</a></h2>
<p style = "font-size : 15px;">Sequential error correction using <code>Gradient Boosting</code> to maximize predictive power.</p>
<p>The model is configured with the following attributes to balance learning and stability:
</p>
<ul style = "font-size : 15px;">
    <li><code>n_estimators=50</code>: This defines the number of boosting stages. We are using a sequence of 50 trees to improve our predictions.</li>
    <li><code>learning_rate=0.05</code>: This controls how much each tree contributes to the final result. A smaller value makes the model learn more slowly and carefully, which often leads to better performance on new data.</li>
    <li><code>max_depth=5</code>: This limits the complexity of each individual tree. By restricting the trees to 5 levels, we prevent them from becoming too specialized and overfitting.</li>
    

In [16]:
gb = GradientBoostingClassifier(n_estimators=50, learning_rate=0.05, max_depth=5,random_state=42)
gb.fit(X_train_final, y_train_final)

y_pred_gb = gb.predict(X_test_final)
print("Gradient Boosting Accuracy:", accuracy_score(y_test_final, y_pred_gb))
print(classification_report(y_test_final, y_pred_gb))

Gradient Boosting Accuracy: 0.7721153846153846
              precision    recall  f1-score   support

           0       0.72      0.61      0.66       379
           1       0.79      0.87      0.83       661

    accuracy                           0.77      1040
   macro avg       0.76      0.74      0.74      1040
weighted avg       0.77      0.77      0.77      1040



In [ ]:
gb2=GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42)
gb2.fit(X_train_final, y_train_final)

y_pred_gb2 = gb2.predict(X_test_final)
print("Gradient Boosting Accuracy:", accuracy_score(y_test_final, y_pred_gb2))
print(classification_report(y_test_final, y_pred_gb2))

Gradient Boosting Accuracy: 0.7730769230769231
              precision    recall  f1-score   support

           0       0.72      0.61      0.66       379
           1       0.80      0.86      0.83       661

    accuracy                           0.77      1040
   macro avg       0.76      0.74      0.75      1040
weighted avg       0.77      0.77      0.77      1040



In [ ]:
#1. Define the model
# GB = GradientBoostingClassifier()

#2. Define the "Grid" of parameters
# param_grid = {
#    'n_estimators': [50, 100, 150, 200, 250, 300, 500],
#    'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.1, 0.5],
#    'max_depth': [3, 5, 7, 8, None]
#}

#3. Set up the Grid Search 
# cv=5 means 5 folds.
#grid_search = GridSearchCV(
#    estimator=GB,
#    param_grid=param_grid,
#    cv=5, 
#    scoring='accuracy',
#    n_jobs=-1,
#    verbose=0 # Set to 0 to avoid conflicting with tqdm output
# )


# len(ParameterGrid) gets the number of combinations. We multiply by cv (5).
#total_iterations = len(ParameterGrid(param_grid)) * 5 

#print(f"Total fits to perform: {total_iterations}")

#4. Run Search with Progress Bar
# Ensure X_train_final and y_train_final are defined before this step
#with tqdm_joblib(tqdm(desc="Tuning Gradient Boosting", total=total_iterations)) as progress_bar:
#    grid_search.fit(X_train_final, y_train_final)

# 5. Results
#print("\n--- Best Results ---")
#print(f"Best Params: {grid_search.best_params_}")
#print(f"Best CV Accuracy: {grid_search.best_score_:.4f}")

# Final Evaluation
#best_gb = grid_search.best_estimator_ # Renamed from best_adb to best_gb
#y_pred_tuned = best_gb.predict(X_test_final)

#print("Validation Accuracy:", accuracy_score(y_test_final, y_pred_tuned))
#print(classification_report(y_test_final, y_pred_tuned))

Total fits to perform: 1050


Tuning Gradient Boosting: 100%|██████████| 1050/1050 [03:49<00:00,  4.58it/s]


--- Best Results ---
Best Params: {'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 300}
Best CV Accuracy: 0.7710
Validation Accuracy: 0.7788461538461539
              precision    recall  f1-score   support

           0       0.73      0.63      0.67       379
           1       0.80      0.87      0.83       661

    accuracy                           0.78      1040
   macro avg       0.77      0.75      0.75      1040
weighted avg       0.78      0.78      0.77      1040



Best Params: {'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 300}

<hr style = "border: 3px solid #0EE071;"> <h2 style = "color : #0EE071;">5. K-Nearest Neighbors (KNN)</a></h2> <p style = "font-size : 15px;">Instance-based learning using <code>KNN</code> to classify data points based on the proximity and characteristics of their neighbors.</p> <p>The model is configured with the following attributes to refine its proximity-based logic: </p> <ul style = "font-size : 15px;"> <li><code>n_neighbors=10</code>: This sets the number of nearby data points the model looks at to make a decision. Using 10 neighbors helps smooth out noise in the dataset.</li> <li><code>metric='manhattan'</code>: This defines the distance calculation as the sum of absolute differences (L1 norm). It measures distance along axes at right angles, often more effective than Euclidean distance in high-dimensional spaces.</li> <li><code>weights='distance'</code>: This ensures that closer neighbors have a stronger influence on the classification than those further away, making the model more sensitive to local patterns.</li> </ul>

In [20]:
knn = KNeighborsClassifier(n_neighbors=10,metric='manhattan',weights='distance')
knn.fit(X_train_final, y_train_final)

y_pred_knn = knn.predict(X_test_final)
print("KNN Accuracy:", accuracy_score(y_test_final, y_pred_knn))
print(classification_report(y_test_final, y_pred_knn))

KNN Accuracy: 0.7692307692307693
              precision    recall  f1-score   support

           0       0.73      0.58      0.65       379
           1       0.78      0.88      0.83       661

    accuracy                           0.77      1040
   macro avg       0.76      0.73      0.74      1040
weighted avg       0.77      0.77      0.76      1040



<p style = "font-size : 15px;">Here we do a <code>GridSearchCV</code> to find the more suitable parameters for future models, tuning parameters such as <code>'algorithm'</code> and  <code> 'Leaf_size'</code>.</p>

In [ ]:
# 1. Define the model
#KNN = KNeighborsClassifier()

# 2. Define the "Grid" of parameters to test
# We will test 3 different learning rates and 3 different estimator counts.
# This results in 3 x 3 = 9 total combinations.
# param_grid = {
#    'n_neighbors': [3, 5, 7, 9, 11, 15],  # The main setting: how many neighbors to look at
#
#    'weights': ['uniform', 'distance'],                 # specific weight strategy
#    
#
#    'metric': ['minkowski', 'euclidean', 'manhattan'],  # How to measure distance
#    'p': [1, 2],                                        # Power parameter (1=Manhattan, 2=Euclidean)
#    
 #   # --- algorithm settings (affects speed, not usually accuracy) ---
#    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'], # How to find neighbors
#    'leaf_size': [10, 30, 60]
#}

# 3. Set up the Grid Search
# cv=5 means "Cross Validation": it splits data into 5 parts to verify scores 5 times.
# n_jobs=-1 uses all your computer's CPU cores to run faster.
#grid_search = GridSearchCV(
#    estimator=KNN, 
#    param_grid=param_grid, 
#    cv=5, 
#    scoring='accuracy', 
#    n_jobs=-1,
#    verbose=2
#)

# 4. Fit the search to your data
#grid_search.fit(X_train_final, y_train_final)

# 5. Get the results
#print("Best Parameters found:", grid_search.best_params_)
#print("Best Accuracy Score:", grid_search.best_score_)


Fitting 5 folds for each of 864 candidates, totalling 4320 fits
Best Parameters found: {'algorithm': 'auto', 'leaf_size': 10, 'metric': 'minkowski', 'n_neighbors': 15, 'p': 1, 'weights': 'distance'}
Best Accuracy Score: 0.7502976603721188


Best Parameters found: {'algorithm': 'auto', 'leaf_size': 10, 'metric': 'minkowski', 'n_neighbors': 15, 'p': 1, 'weights': 'distance'}

<hr style = "border: 3px solid #0EE071;"> <h2 style = "color : #0EE071;">6. Decision Tree</a>
</h2> <p style = "font-size : 15px;">A flow-chart-like structure using <code>Decision Tree</code> to map features to outcomes through hierarchical splits.</p> 
<p>The model is configured with the following attributes to maintain a balance between detail and generalization: </p> 
<ul style = "font-size : 15px;"> 
<li><code>max_depth=5</code>: This limits how deep the tree can grow. Restricting the depth to 5 levels prevents the model from creating overly complex rules that might only apply to the training data (overfitting).</li>
 <li><code>criterion='gini'</code>: This uses the Gini Impurity to measure the "purity" of a node. It helps the algorithm choose the best feature and split point to separate the classes most effectively at each step.</li>
 

In [19]:

DT=DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
DT.fit(X_train_final, y_train_final)
y_pred_DT = DT.predict(X_test_final)
print("DT Accuracy:", accuracy_score(y_test_final, y_pred_DT))
print(classification_report(y_test_final, y_pred_DT))

DT Accuracy: 0.7461538461538462
              precision    recall  f1-score   support

           0       0.65      0.65      0.65       379
           1       0.80      0.80      0.80       661

    accuracy                           0.75      1040
   macro avg       0.73      0.72      0.73      1040
weighted avg       0.75      0.75      0.75      1040



In [22]:
DT2=DecisionTreeClassifier(max_depth=10, criterion='gini', random_state=42)
DT2.fit(X_train_final, y_train_final)
y_pred_dt2 = DT2.predict(X_test_final) 
print("DT2 Accuracy:", accuracy_score(y_test_final, y_pred_dt2))
print(classification_report(y_test_final, y_pred_dt2))

DT2 Accuracy: 0.7230769230769231
              precision    recall  f1-score   support

           0       0.62      0.63      0.62       379
           1       0.78      0.78      0.78       661

    accuracy                           0.72      1040
   macro avg       0.70      0.70      0.70      1040
weighted avg       0.72      0.72      0.72      1040



In [ ]:
#DTg = DecisionTreeClassifier()

# 2. Define the "Grid" of parameters to test

#param_grid = {
#    'max_depth': [5, 10, 15, 20, None],  # The main setting: how deep the tree goes
#
#    'criterion': ['gini', 'entropy', 'log_loss'],                 # specific weight strategy
#
#    'min_samples_split': [2, 5, 10],  # Minimum samples to split a node
#    'min_samples_leaf': [1, 2, 4],    # Minimum samples at a leaf node
#    
#    # --- splitter settings ---
#    'splitter': ['best', 'random']    # How to choose splits
#}

# 3. Set up the Grid Search
# cv=5 means "Cross Validation": it splits data into 5 parts to verify scores 5 times.
# n_jobs=-1 uses all your computer's CPU cores to run faster.
#grid_search = GridSearchCV(
#    estimator=DTg, 
#    param_grid=param_grid, 
#    cv=5, 
#    scoring='accuracy', 
#    n_jobs=-1,
#    verbose=2
#)

# 4. Fit the search to your data
# (Make sure to use your training data, not validation data here)
#grid_search.fit(X_train_final, y_train_final)

# 5. Get the results
#print("Best Parameters found:", grid_search.best_params_)
#print("Best Accuracy Score:", grid_search.best_score_)



Fitting 5 folds for each of 270 candidates, totalling 1350 fits
Best Parameters found: {'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 5, 'splitter': 'random'}
Best Accuracy Score: 0.7349040775710451


<hr style = "border: 3px solid #0EE071;">
<h2 style = "color : #0EE071;"> 7. Initial MLP Classifier (Neural Network)</h2>
<p style = "font-size : 15px;">
    In this section, we implement the <b>MLPClassifier</b> (Multi-layer Perceptron), which is a type of artificial neural network. Unlike linear models, the MLP can learn complex, non-linear relationships by passing data through "hidden layers" of interconnected neurons.
</p>
<p style = "font-size : 15px;">
    This specific model is configured with the following key attributes:
</p>
<ul style = "font-size : 15px;">
    <li><code>hidden_layer_sizes=(100, 50)</code>: This defines the architecture of our neural network. It consists of two hidden layers: the first has 100 neurons and the second has 50 neurons. This allows the model to process information in increasingly abstract ways.</li>
   <li><code>solver='lbfgs'</code>: This is the "mathematical engine" or optimization algorithm that the model uses to find the best possible weights. We selected <code>lbfgs</code> because it is particularly efficient at finding the optimal solution quickly for small to medium-sized datasets like ours.<b>(This will be utilized in further versions of this model)</b></li>
</ul>
<p style = "font-size : 15px;">
    Once trained with <code>fit()</code>, we evaluate the model using <code>accuracy_score</code> and a <code>classification_report</code> to see how well it balances precision and recall across our classes.
</p>

In [ ]:
MLP1 = MLPClassifier(hidden_layer_sizes=(100, 50), random_state = 42)
MLP1.fit(X_train_final,y_train_final)
y_pred_MLP=MLP1.predict(X_test_final)
print("MLPClassifier Accuracy:", accuracy_score(y_test_final, y_pred_MLP))
print(classification_report(y_test_final, y_pred_MLP))

MLPClassifier Accuracy: 0.7692307692307693
              precision    recall  f1-score   support

           0       0.71      0.61      0.66       379
           1       0.79      0.86      0.83       661

    accuracy                           0.77      1040
   macro avg       0.75      0.74      0.74      1040
weighted avg       0.77      0.77      0.76      1040



In [24]:
MLP2= MLPClassifier(hidden_layer_sizes=(100, 50, 25), random_state = 15)
MLP2.fit(X_train_final,y_train_final)
y_pred_MLP2=MLP2.predict(X_test_final)
print("MLPClassifier Accuracy:", accuracy_score(y_test_final, y_pred_MLP2))
print(classification_report(y_test_final, y_pred_MLP2))

MLPClassifier Accuracy: 0.7288461538461538
              precision    recall  f1-score   support

           0       0.63      0.62      0.63       379
           1       0.78      0.79      0.79       661

    accuracy                           0.73      1040
   macro avg       0.71      0.71      0.71      1040
weighted avg       0.73      0.73      0.73      1040



In [25]:
MLP3=MLPClassifier(solver='lbfgs', random_state = 15)  # Good for smaller datasets)
MLP3.fit(X_train_final,y_train_final)
y_pred_MLP3=MLP3.predict(X_test_final)
print("MLPClassifier Accuracy:", accuracy_score(y_test_final, y_pred_MLP3))
print(classification_report(y_test_final, y_pred_MLP3))

MLPClassifier Accuracy: 0.7615384615384615
              precision    recall  f1-score   support

           0       0.70      0.60      0.65       379
           1       0.79      0.86      0.82       661

    accuracy                           0.76      1040
   macro avg       0.75      0.73      0.73      1040
weighted avg       0.76      0.76      0.76      1040



In [ ]:
#MLPg = MLPClassifier()

# 2. Define the "Grid" of parameters to test

#param_grid = {
#    'hidden_layer_sizes': [(50,), (100,), (100, 50), (100, 50, 25)],  # Different layer sizes
#
 #   'activation': ['relu', 'tanh', 'logistic'],                        # Activation functions
#
 #   'solver': ['adam', 'sgd', 'lbfgs'],                                # Optimization algorithms
#
#    'alpha': [0.0001, 0.001, 0.01],                                   # Regularization strength
#
#    'learning_rate': ['constant', 'adaptive'],                         # Learning rate schedule
#}

# 3. Set up the Grid Search.
# n_jobs=-1 uses all your computer's CPU cores to run faster.
#grid_search = GridSearchCV(
 #   estimator=MLPg, 
 #   param_grid=param_grid, 
 #   cv=5, 
#    scoring='accuracy', 
 #   n_jobs=-1,
 #   verbose=2
#)

# 4. Fit the search to your data
# (Make sure to use your training data, not validation data here)
#grid_search.fit(X_train_final, y_train_final)
# 5. Get the results
#print("Best Parameters found:", grid_search.best_params_)
#print("Best Accuracy Score:", grid_search.best_score_)



Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best Parameters found: {'activation': 'tanh', 'alpha': 0.0001, 'hidden_layer_sizes': (50,), 'learning_rate': 'constant', 'solver': 'adam'}
Best Accuracy Score: 0.7618453785985374


<p>Best Parameters found: {'activation': 'tanh', 'alpha': 0.0001, 'hidden_layer_sizes': (50,), 'learning_rate': 'constant', 'solver': 'adam'}</p>

<hr style = "border: 3px solid #0EE071;">
<h2 style = "color : #0EE071;"> 8. AdaBoost Classifier</h2>
<p style = "font-size : 15px;">
    In this section, we implement the <code>AdaBoostClassifier</code> to improve our model's performance through sequential learning. We explore boosting using different base estimators, including <code>RandomForestClassifier</code> and <code>LogisticRegression</code>. Using <code>GridSearchCV</code>, we optimize the <code>n_estimators</code> and <code>learning_rate</code>, while also fine-tuning the internal complexity of the base estimators to achieve the best predictive accuracy.
</p>
<p style = "font-size : 15px;">
    The model is configured with the following key attributes:
</p>
<ul style = "font-size : 15px;">
    <li><code>estimator=RandomForestClassifier()</code>: This tells AdaBoost to use a full Random Forest as its internal learner, allowing it to capture more complex patterns in each boosting round.</li>
    <li><code>n_estimators=50</code>: The algorithm will perform 50 boosting iterations, sequentially refining the model's focus on previously misclassified instances.</li>
    <li><code>learning_rate=0.5</code>: This determines the "speed" of learning. A value of 0.5 provides a moderate pace, helping to prevent the model from overreacting to outliers in the data.</li>
    <li><code>random_state=42</code>: Ensures that the random selection of features and samples is consistent, making the results of this complex ensemble fully reproducible.</li>

<p>For the following variations of <code>AdaBoost</code> we start experimenting with various values before doing <code>GridSearch</code>

In [27]:
ADB= AdaBoostClassifier(estimator= RandomForestClassifier(), n_estimators=50, learning_rate=0.5, random_state=42)
ADB.fit(X_train_final, y_train_final)
y_pred_adb = ADB.predict(X_test_final)
print("AdaBoost Classifier Accuracy:", accuracy_score(y_test_final, y_pred_adb))
print(classification_report(y_test_final, y_pred_adb))

AdaBoost Classifier Accuracy: 0.7798076923076923
              precision    recall  f1-score   support

           0       0.73      0.63      0.68       379
           1       0.80      0.87      0.83       661

    accuracy                           0.78      1040
   macro avg       0.77      0.75      0.75      1040
weighted avg       0.78      0.78      0.78      1040



In [28]:
ADB2= AdaBoostClassifier(estimator= RandomForestClassifier(),n_estimators=100, learning_rate=1.0, random_state=42)
ADB2.fit(X_train_final, y_train_final)
y_pred_adb2 = ADB2.predict(X_test_final)
print("AdaBoost Classifier 2 Accuracy:", accuracy_score(y_test_final, y_pred_adb2))
print(classification_report(y_test_final, y_pred_adb2))

AdaBoost Classifier 2 Accuracy: 0.7798076923076923
              precision    recall  f1-score   support

           0       0.73      0.63      0.68       379
           1       0.80      0.87      0.83       661

    accuracy                           0.78      1040
   macro avg       0.77      0.75      0.75      1040
weighted avg       0.78      0.78      0.78      1040



<p>Here we decided to experiment with other type of estimators such as <code>Logistic Regression</code> to see how accuracy changed</p>

In [29]:
ADB3= AdaBoostClassifier(estimator= LogisticRegression(),n_estimators=150, learning_rate=0.25, random_state=15)
ADB3.fit(X_train_final, y_train_final)
y_pred_adb3 = ADB3.predict(X_test_final)
print("AdaBoost Classifier 3 Accuracy:", accuracy_score(y_test_final, y_pred_adb3))
print(classification_report(y_test_final, y_pred_adb3))

AdaBoost Classifier 3 Accuracy: 0.7307692307692307
              precision    recall  f1-score   support

           0       0.63      0.63      0.63       379
           1       0.79      0.79      0.79       661

    accuracy                           0.73      1040
   macro avg       0.71      0.71      0.71      1040
weighted avg       0.73      0.73      0.73      1040



In [30]:
ADB4= AdaBoostClassifier(estimator= RandomForestClassifier(), n_estimators=150, learning_rate=0.25, random_state=15)
ADB4.fit(X_train_final, y_train_final)
y_pred_adb4 = ADB4.predict(X_test_final)
print("AdaBoost Classifier 4 Accuracy:", accuracy_score(y_test_final, y_pred_adb4))
print(classification_report(y_test_final, y_pred_adb4))

AdaBoost Classifier 4 Accuracy: 0.7875
              precision    recall  f1-score   support

           0       0.74      0.65      0.69       379
           1       0.81      0.87      0.84       661

    accuracy                           0.79      1040
   macro avg       0.77      0.76      0.76      1040
weighted avg       0.78      0.79      0.78      1040



<hr style = "border: 3px solid #0EE071;">
<h3 style = "color : #0EE071;"> 8.1. AdaBoost Hyperparameter Optimization</h3>
<p style = "font-size : 15px;">
    To further refine the performance of the <code>AdaBoostClassifier</code>, we perform a comprehensive search across a broad <code>parameter_space</code>. Unlike standard boosting which typically uses decision trees, we evaluate four distinct algorithms as the base <code>estimator</code>:
</p>
<ul style = "font-size : 15px;">
    <li><b>Base Learners:</b> <code>RandomForestClassifier</code>, <code>DecisionTreeClassifier</code>, <code>MLPClassifier</code>, and <code>KNeighborsClassifier</code>.</li>
    <li><b>Boosting Dynamics:</b> We test combinations of <code>n_estimators</code> (up to 200) and <code>learning_rate</code> (ranging from 0.1 to 0.8) to find the best trade-off between speed and accuracy.</li>
    <li><b>Algorithms:</b> We compare <code>SAMME</code> and <code>SAMME.R</code> to determine the most effective boosting algorithm for our data.</li>
</ul>
<p style = "font-size : 15px;">
    We utilize <code>GridSearchCV</code> with <code>n_jobs=-1</code> to parallelize the process and identify the <code>best_estimator_</code> based on cross-validation accuracy.
</p>

In [ ]:
#parameter_space_adaboost = {'estimator': [RandomForestClassifier(), DecisionTreeClassifier(), MLPClassifier(), KNeighborsClassifier()],
 #                           'n_estimators': [20, 50, 80,100,150,200], 
 #                           'learning_rate': [0.1,0.3,0.4, 0.5, 0.6, 0.8],
 #                           'algorithm': ['SAMME', 'SAMME.R'],
 #                           'random_state': [42]}

In [ ]:
#grid_search_adaboost = GridSearchCV(AdaBoostClassifier(), parameter_space_adaboost, scoring='accuracy',verbose=10,n_jobs=-1).fit(X_train_final, y_train_final)
#best_model = grid_search_adaboost.best_estimator_
#y_pred_best = best_model.predict(X_test_final)
#print("Best Parameters found:", grid_search_adaboost.best_params_)
#print("Best Accuracy Score:", grid_search_adaboost.best_score_)

Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Best Parameters found: {'algorithm': 'SAMME', 'estimator': RandomForestClassifier(), 'learning_rate': 0.1, 'n_estimators': 20, 'random_state': 42}
Best Accuracy Score: 0.777485131444969


Best Parameters found: {'algorithm': 'SAMME', 'estimator': RandomForestClassifier(), 'learning_rate': 0.1, 'n_estimators': 20, 'random_state': 42}

<hr style = "border: 3px solid #0EE071;">
<h2  style = "color : #0EE071;">9. Ensemble Methods</h2>
<p style = "font-size : 15px;">Combining multiple base learners (RF, GB, etc.) using a <code>StackingClassifier</code> with a Logistic Regression meta-learner for final predictions.</p>

In [ ]:
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1) 

adb = AdaBoostClassifier(
    estimator=rf_base,
    random_state=42,
    algorithm='SAMME' # 'SAMME' is required for some RF configurations in newer sklearn
)

# 3. Define the Parameter Grid 
# USE "estimator__" to tune the Random Forest inside the AdaBoost
param_grid = {
    # AdaBoost Parameters (Outer Loop)
    'n_estimators': [10, 30, 50],       # How many Random Forests to boost sequentially
    'learning_rate': [0.1, 0.5, 1.0],   # How much to correct errors

    # Random Forest Parameters (Inner Loop) 
    # Access via 'estimator__' prefix
    'estimator__n_estimators': [10, 20],# Keep this LOW. 50 AdaBoost * 20 RF = 1000 trees total
    'estimator__max_depth': [3, 5, 10], # Shallow trees usually boost better than deep ones
    'estimator__min_samples_split': [2, 5]
}

#4. Initialize Grid Search 
grid_search = GridSearchCV(
    adb, 
    param_grid, 
    cv=3, 
    scoring='accuracy', 
    n_jobs=-1,  # Uses parallel processing
    verbose=0
)

# 5. Run with Progress Bar 
total_iterations = len(ParameterGrid(param_grid)) * 3 # 3 CV folds

print(f"Total models to train: {total_iterations}")

with tqdm_joblib(tqdm(desc="Tuning AdaBoost(RF)", total=total_iterations)) as progress_bar:
    grid_search.fit(X_train_final, y_train_final)

#  6. Results
print("\n--- Best Results ---")
print(f"Best Params: {grid_search.best_params_}")
print(f"Best CV Accuracy: {grid_search.best_score_:.4f}")

# Final Evaluation
best_adb = grid_search.best_estimator_
y_pred_tuned = best_adb.predict(X_test_final)
from sklearn.metrics import classification_report, accuracy_score
print("Test Accuracy:", accuracy_score(y_test_final, y_pred_tuned))
print(classification_report(y_test_final, y_pred_tuned))

Total models to train: 324


Tuning AdaBoost(RF): 1524it [01:18, 19.36it/s]                        



--- Best Results ---
Best Params: {'estimator__max_depth': 5, 'estimator__min_samples_split': 5, 'estimator__n_estimators': 20, 'learning_rate': 0.1, 'n_estimators': 30}
Best CV Accuracy: 0.7688
Test Accuracy: 0.7644230769230769
              precision    recall  f1-score   support

           0       0.72      0.58      0.64       379
           1       0.78      0.87      0.82       661

    accuracy                           0.76      1040
   macro avg       0.75      0.73      0.73      1040
weighted avg       0.76      0.76      0.76      1040



<h3  style = "color : #0EE071;"> 9.1. Final Stacking Classifier Implementation</h3>
<p style = "font-size : 15px;">
    In this final stage of modeling, we implement a <code>StackingClassifier</code> to leverage the strengths of multiple algorithms. The architecture consists of two layers:
</p>
<ul style = "font-size : 15px;">
    <li><b>Base Estimators:</b> We utilize <code>AdaBoostClassifier</code> (depth 3), <code>GradientBoostingClassifier</code> (with <code>subsample=0.8</code> to combat overfitting), and <code>RandomForestClassifier</code>.</li>
    <li><b>Meta-Learner:</b> A <code>LogisticRegression</code> model acts as the final aggregator, learning how to best combine the predictions from the base layer.</li>
</ul>
<p style = "font-size : 15px;">
    We utilize a 5-fold cross-validation (<code>cv=5</code>) strategy within the stack to generate robust out-of-fold predictions for the meta-learner. Finally, we evaluate the performance on the test set and perform an overfitting diagnosis by comparing training and validation accuracies.
</p>

In [ ]:

# 1. Define the Base Estimators

# AdaBoost: Base estimator is a Tree with max_depth=3 to minimize overfitting
ada_base = DecisionTreeClassifier(max_depth=3, random_state=42)
ada_model = AdaBoostClassifier(
    estimator=ada_base,
    learning_rate=0.1, 
    n_estimators=150, 
    random_state=42
)

# Gradient Boosting: Depth limited to 3 to control complexity and overfitting and subsample to 0.8 to reduce overfitting as well
gb_model = GradientBoostingClassifier(
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    n_estimators=200,
    random_state=42
)

# Random Forest: min_samples_split=5
rf_model = RandomForestClassifier(
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=5,
    n_estimators=200,
    random_state=42
)

#  2. Define the Final Estimator (Meta-learner) 
final_layer = LogisticRegression(
    C=1.0, 
    solver='lbfgs', 
    random_state=42
)

#  3. Create the Stacking Classifier 
estimators_list = [
    ('adabosting', ada_model),
    ('gb', gb_model),
    ('rf', rf_model)
]

stacking_final = StackingClassifier(
    estimators=estimators_list,
    final_estimator=final_layer,
    cv=5,       # Uses 5-fold cross-validation to generate inputs for the meta-learner
    n_jobs=-1   # Uses all CPU cores
)

#  4. Train and Evaluate 
print("Training Stacking Classifier...")
stacking_final.fit(X_train_final, y_train_final)

# Predict on test set
y_pred_stack_final = stacking_final.predict(X_test_final)

#  5. Output Metrics 
print("\n--- Final Stacking Model Performance ---")
print(f"Test Accuracy: {accuracy_score(y_test_final, y_pred_stack_final):.4f}")
print(classification_report(y_test_final, y_pred_stack_final))

# Check for Overfitting on Training Data
print("\n--- Training Check (Overfitting Diagnosis) ---")
y_train_pred = stacking_final.predict(X_train_final)
print(f"Training Accuracy:   {accuracy_score(y_train_final, y_train_pred):.4f}")

Training Stacking Classifier...
(Note: This may take a moment because GradientBoosting max_depth is set to None)

--- Final Stacking Model Performance ---
Test Accuracy: 0.7817
              precision    recall  f1-score   support

           0       0.74      0.63      0.68       379
           1       0.80      0.87      0.84       661

    accuracy                           0.78      1040
   macro avg       0.77      0.75      0.76      1040
weighted avg       0.78      0.78      0.78      1040


--- Training Check (Overfitting Diagnosis) ---
Training Accuracy:   0.8605


<p>Here we did some <code>GridSearches</code> to verify if we could find better parameters all on the same model of <code>Stacking</code> and verify overfitting.</p>

In [ ]:

#  1. Data Preparation
# Scale features for KNN
#scaler = StandardScaler()
#X_train_scaled = scaler.fit_transform(X_train_final)
#X_val_scaled = scaler.transform(X_val_final)
#
#  3. Initialize Stack (The Structure) 
#estimators = [
#    ('knn', KNeighborsClassifier()),
#    ('rf', RandomForestClassifier(random_state=42)),
#    ('gb', GradientBoostingClassifier(random_state=42))
#]

#stack = StackingClassifier(
#    estimators=estimators,
#    final_estimator=LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42),
#    passthrough=True, 
#    cv=5,
#    n_jobs=1  # Important: Keep 1 here so GridSearch controls the parallelism
#)

#  4. Define Parameter Grid 
# Cartesian Product: 3 (KNN) * 2 (RF) * 2 (RF) * 2 (GB) * 2 (GB) * 3 (Final) = 144 combinations
#param_grid = {
#    'knn__n_neighbors': [5, 7, 9],
#    'rf__n_estimators': [100, 200],
#   'rf__max_depth': [5, 10],            # Restricted depth
#   'gb__n_estimators': [100, 150],
#    'gb__max_depth': [3, 5],             # Restricted depth
#    'final_estimator__C': [0.1, 1.0, 10.0]
#}

#  5. Run Grid Search with Progress Bar 
#grid = GridSearchCV(
#    estimator=stack,
#    param_grid=param_grid,
#    cv=5,
#    scoring='accuracy',
#    n_jobs=-1,  # Use all cores
#    verbose=0   # Turn off text output so it doesn't mess up the progress bar
#)

# Calculate total tasks = (Number of Combinations) * (Number of Folds)
#total_models = len(ParameterGrid(param_grid)) * 5 

#print(f"Total models to train: {total_models}")
#print("Starting Grid Search...")

#with tqdm_joblib(tqdm(desc="Grid Search Progress", total=total_models)):
#   grid.fit(X_train_scaled, y_train_final)

# 6. Results 
#print(f"\nBest Parameters: {grid.best_params_}")
#print(f"Best CV Score: {grid.best_score_:.4f}")

# Final Validation
#best_model = grid.best_estimator_
#y_pred_grid = best_model.predict(X_val_scaled)

#print("\n--- Validation Results ---")
#print(f"Validation Accuracy: {accuracy_score(y_test_final, y_pred_grid):.4f}")
#print(classification_report(y_test_final, y_pred_grid))

# Overfitting Check
#train_acc = accuracy_score(y_train_final, best_model.predict(X_train_scaled))
#print(f"\nTraining Accuracy (Check): {train_acc:.4f}")

#gap = train_acc - accuracy_score(y_test_final, y_pred_grid)
#if gap > 0.1:
#    print(f" Warning: Gap is {gap:.2f}. Mild overfitting.")
#else:
#    print(f" Success: Gap is {gap:.2f}. Model is robust.")

In [2]:
#estimators = [
#    ('rf', RandomForestClassifier(n_jobs=-1, random_state=42)),
#    ('gb', GradientBoostingClassifier(random_state=42)), 
#    # Note: Using a Random Forest inside AdaBoost is very heavy. 
#    # We limit the inner RF to 10 trees to keep it faster.
#    ('adabosting', AdaBoostClassifier(estimator=RandomForestClassifier(n_estimators=10, n_jobs=-1), random_state=42))
#]
#
#  3. Define the Stacking Classifier 
#clf = StackingClassifier(
#    estimators=estimators,
#    final_estimator=LogisticRegression(),
#    cv=5,
#    n_jobs=-1,
#    passthrough=False
#)

#  4. Define the Parameter Grid 
#param_grid = {
#    #  Random Forest Parameters 
#    'rf__n_estimators': [100,150, 300],
#    'rf__max_depth': [3, 10],
#    'rf__min_samples_split': [2, 5],
#
#    #  Gradient Boosting Parameters 
#    'gb__n_estimators': [100,150, 200],
#    'gb__learning_rate': [0.01,0.05, 0.1],
#    'gb__max_depth': [3, 5],
#
#    #  AdaBoost Parameters (Newly Added) 
#    # We tune the boosting process itself:
#    'adabosting__n_estimators': [100,150,200],      # How many boosting rounds
#    'adabosting__learning_rate': [0.1, 1.0],    # Weight applied to each classifier
#    
#    # Options for the Random Forest inside AdaBoost:
#    'adabosting__estimator__max_depth': [3, 5], 
#
#    #  Final Estimator Parameters 
#    'final_estimator__C': [0.1, 1.0],
 #   'final_estimator__solver': ['lbfgs']
#}

#  5. Initialize Grid Search 
#grid_search = GridSearchCV(
#    clf, 
#    param_grid, 
#    cv=3,                
#    scoring='accuracy',  
#    n_jobs=-1,           
#    verbose=0 
#)

#  6. Run with Progress Bar 
#number_of_combinations = len(ParameterGrid(param_grid))
#cv_folds = 3  
#total_iterations = number_of_combinations * cv_folds

#print(f"Total fits to perform: {total_iterations}")

# Ensure X_train_final and y_train_final are defined in your environment
#with tqdm_joblib(tqdm(desc="Tuning Hyperparameters", total=total_iterations)) as progress_bar:
#    grid_search.fit(X_train_final, y_train_final)

#print("\n--- Search Complete ---")
#print(f"Best Params: {grid_search.best_params_}")
#print(f"Best Score: {grid_search.best_score_}")

In [ ]:
#start_time = time.time()


# 1. TUNE RANDOM FOREST

#print("--- Tuning Random Forest (Step 1/4) ---")
#rf = RandomForestClassifier(random_state=42, n_jobs=-1)
#param_grid_rf = {
#    'n_estimators': [100, 200, 300],
#    'max_depth': [None, 10],
#    'min_samples_split': [2, 5]
#}
#grid_rf = GridSearchCV(rf, param_grid_rf, cv=3, scoring='accuracy', n_jobs=-1, verbose=0)
#grid_rf.fit(X_train_final, y_train_final)
##best_rf = grid_rf.best_estimator_


# 2. TUNE GRADIENT BOOSTING

#print("--- Tuning Gradient Boosting (Step 2/4) ---")
#gb = GradientBoostingClassifier(random_state=42)
#param_grid_gb = {
 #   'n_estimators': [100, 200],
 #   'learning_rate': [0.05, 0.1],
 #   'max_depth': [3, 5, None]
#}
#grid_gb = GridSearchCV(gb, param_grid_gb, cv=3, scoring='accuracy', n_jobs=-1, verbose=0)
#grid_gb.fit(X_train_final, y_train_final)
#best_gb = grid_gb.best_estimator_


# 3. TUNE ADABOOST

#print("--- Tuning AdaBoost (Step 3/4) ---")
# Using a lighter base estimator for speed
#ada = AdaBoostClassifier(estimator=RandomForestClassifier(n_estimators=10, max_depth=5, n_jobs=-1), random_state=42)
#param_grid_ada = {
#    'n_estimators': [50, 100, 150],
#    'learning_rate': [0.1, 1.0]
#}
#grid_ada = GridSearchCV(ada, param_grid_ada, cv=3, scoring='accuracy', n_jobs=-1, verbose=0)
#grid_ada.fit(X_train_final, y_train_final)
#best_ada = grid_ada.best_estimator_


# 4. TUNE STACKING META-LEARNER

#print("--- Tuning Final Stacker (Step 4/4) ---")
#estimators_optimized = [
#    ('rf', best_rf),
#    ('gb', best_gb),
#    ('adabosting', best_ada) # Naming it 'adabosting' to match your requested format
#]
#clf_stack = StackingClassifier(
#    estimators=estimators_optimized,
#    final_estimator=LogisticRegression(solver='lbfgs', max_iter=1000),
#    cv=5,
#    n_jobs=-1,
#    passthrough=False 
#)
#param_grid_meta = {
#    'final_estimator__C': [0.1, 1.0, 10.0],
#    'final_estimator__solver': ['lbfgs']
#}

#grid_stack = GridSearchCV(clf_stack, param_grid_meta, cv=3, scoring='accuracy', n_jobs=-1, verbose=0)
#grid_stack.fit(X_train_final, y_train_final)


# 5. CONSOLIDATE AND PRINT RESULTS


# Create a single dictionary that looks like the result of one giant GridSearch
#full_best_params = {}

# Add RF params (add 'rf__' prefix)
#for key, val in grid_rf.best_params_.items():
#    full_best_params[f'rf__{key}'] = val

# Add GB params (add 'gb__' prefix)
#for key, val in grid_gb.best_params_.items():
#    full_best_params[f'gb__{key}'] = val

# Add AdaBoost params (add 'adabosting__' prefix)
#for key, val in grid_ada.best_params_.items():
#    full_best_params[f'adabosting__{key}'] = val

# Add Meta params (already has 'final_estimator__' prefix)
#full_best_params.update(grid_stack.best_params_)
#print("\n" + "="*30)
#print(f"Best Params: {full_best_params}")
#print(f"Best Score: {grid_stack.best_score_}")
#print("="*30)
#print(f"Total execution time: {round((time.time() - start_time)/60, 2)} minutes")

--- Tuning Random Forest (Step 1/4) ---
--- Tuning Gradient Boosting (Step 2/4) ---
--- Tuning AdaBoost (Step 3/4) ---
--- Tuning Final Stacker (Step 4/4) ---

Best Params: {'rf__max_depth': None, 'rf__min_samples_split': 2, 'rf__n_estimators': 300, 'gb__learning_rate': 0.05, 'gb__max_depth': 3, 'gb__n_estimators': 200, 'adabosting__learning_rate': 0.1, 'adabosting__n_estimators': 50, 'final_estimator__C': 1.0, 'final_estimator__solver': 'lbfgs'}
Best Score: 0.7709873359692855
Total execution time: 0.97 minutes


<hr style = "border: 3px solid #0EE071;">

In [7]:
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-Learn version: {sklearn.__version__}")
print(f"joblib version: {joblib.__version__}")

Pandas version: 2.3.2
Scikit-Learn version: 1.7.1
joblib version: 1.5.2
